# Advanced Data Preparation with Python (Apartment Data)

This notebook demonstrates advanced data preparation techniques with pandas using scraped rental apartment data from Zurich. It is an exercise: complete the empty code cells. The solution is in advanced_data_preparation_apartment_data_solution.ipynb. 
We will cover the following operations:

- Extracting information from raw data using regular expressions
- Counting and removing missing and duplicated values
- Creating new variables using string manipulation
- Discretization and One Hot Encoding
- Scaling and standardization
- Transformations (log, square root, Box-Cox)
- Combining data using .merge()
- Sorting, reshaping and pivoting data

## Libraries and settings

In [1]:
# Libraries
import os
import re
import time
import fnmatch
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

/workspaces/python_data_preparation


## Importing data

In [2]:
# Read the data to a pandas data frame
df = pd.read_csv('./Data/apartments_data_zuerich.csv',
                 sep=',',
                 encoding='utf-8')

# Show first records of data frame
df.head()

,web-scraper-order,web-scraper-start-url,rooms_area_price_raw,address_raw,price_raw,description_raw,text_raw
0,1693998201-1,https://www.immoscout24.ch/de/immobilien/miete...,"3 Zimmer, 49 m², CHF 1441.—","Neuhusstrasse 6, 8630 Rüti ZH, ZH",CHF 1441.—,«Gemütliche Wohnung im Grünen»,"3 Zimmer, 49 m², CHF 1441.—Neuhusstrasse 6, 86..."
1,1693998201-2,https://www.immoscout24.ch/de/immobilien/miete...,"3,5 Zimmer, 65 m², CHF 1850.—","Zürcherstrasse 1, 8173 Neerach, ZH",CHF 1850.—,«Attraktive 3.5-Zimmer-EG-Wohnung in Neerach»,"3,5 Zimmer, 65 m², CHF 1850.—Zürcherstrasse 1,..."
2,1693998201-3,https://www.immoscout24.ch/de/immobilien/miete...,"19 m², CHF 2686.—","Cramerstrasse 8-12, 8004 Zürich, ZH",CHF 2686.—,«Studio Apartment Junior Balcony»,"19 m², CHF 2686.—Cramerstrasse 8-12, 8004 Züri..."
3,1693998201-4,https://www.immoscout24.ch/de/immobilien/miete...,"2 Zimmer, 54 m², CHF 4853.—","Cramerstrasse 8-12, 8004 Zürich, ZH",CHF 4853.—,«2 Bedroom Apartment Senior Balcony»,"2 Zimmer, 54 m², CHF 4853.—Cramerstrasse 8-12,..."
4,1693998201-5,https://www.immoscout24.ch/de/immobilien/miete...,"2 Zimmer, 49 m², CHF 4335.—","Rotachstrasse 33, 8003 Zürich, ZH",CHF 4335.—,«2 Bedroom Apartment Junior Terrace»,"2 Zimmer, 49 m², CHF 4335.—Rotachstrasse 33, 8..."


## Count number of rows and columns in the data frame

In [3]:
#Hier einfach die verlangten Zeilen aus Spick-Jupyter reinkopiert und nichts angepasst
# Dimension (rows, columns)
print("Dimension:", df.shape)

# Number of rows
print("Anzahl Zeilen:", df.shape[0])

# Number of columns
print("Anzahl Spalten:", df.shape[1])



Dimension: (1008, 7)
Anzahl Zeilen: 1008
Anzahl Spalten: 7


## Get data types (raw-format from web scraper)

In [4]:
#Genau aus dem gleichen Codesnipped genommen wie Aufagbe zuvor
# Get data types (note that in pandas, a string is referred to as 'object')
print(df.dtypes)

web-scraper-order        str
web-scraper-start-url    str
rooms_area_price_raw     str
address_raw              str
price_raw                str
description_raw          str
text_raw                 str
dtype: object


## Extract information from raw data using regular expressions (regex)

### Extract number of rooms

In [5]:
# Extract values from 'rooms_area_price_raw' strings
rooms_text = df["rooms_area_price_raw"].str.extract(r"(\d+(?:[.,]\d+)?)\s*Zimmer", expand=False)

# Save as new variable in the pandas data frame
df["rooms"] = pd.to_numeric(rooms_text.str.replace(",", ".", regex=False), errors="coerce")

# Show first records of data frame
display(df[["rooms_area_price_raw", "rooms"]].head())

,rooms_area_price_raw,rooms
0,"3 Zimmer, 49 m², CHF 1441.—",3.0
1,"3,5 Zimmer, 65 m², CHF 1850.—",3.5
2,"19 m², CHF 2686.—",NaN
3,"2 Zimmer, 54 m², CHF 4853.—",2.0
4,"2 Zimmer, 49 m², CHF 4335.—",2.0


### Extract living area

In [14]:
#Spick-Block: Ganze Zahlen und Dezimalzahlen vor einer bestimmten Bezeichnung mit Regex extrahieren
# Extract values from 'rooms_area_price_raw' strings
area_text = df["rooms_area_price_raw"].str.extract(r"(\d+(?:[.,]\d+)?)\s*m²", expand=False)


# Save as new variable in the pandas data frame
df["area"] = pd.to_numeric(area_text.str.replace(",", ".", regex=False), errors="coerce")

# Show first records of data frame
display(df[["rooms_area_price_raw", "area"]].head())


,rooms_area_price_raw,area
0,"3 Zimmer, 49 m², CHF 1441.—",49.0
1,"3,5 Zimmer, 65 m², CHF 1850.—",65.0
2,"19 m², CHF 2686.—",19.0
3,"2 Zimmer, 54 m², CHF 4853.—",54.0
4,"2 Zimmer, 49 m², CHF 4335.—",49.0


### Extract rental price

In [28]:
# Extract values from 'price_raw' strings
number_text = df["price_raw"].str.extract(r"CHF\s*(\d[\d'’]*)", expand=False)


# Save as new variable in the pandas data frame
number_text = number_text.str.replace(r"['’]", "", regex=True)
df["price"] = pd.to_numeric(number_text, errors="coerce")


# Show first records of data frame
display(df[["price_raw", "price"]].head())


,price_raw,price
0,CHF 1441.—,1441.0
1,CHF 1850.—,1850.0
2,CHF 2686.—,2686.0
3,CHF 4853.—,4853.0
4,CHF 4335.—,4335.0


### Get data types of all variables, including the new ones

In [29]:
# Get data types of all variables
print(df.dtypes) 

web-scraper-order            str
web-scraper-start-url        str
rooms_area_price_raw         str
address_raw                  str
price_raw                    str
description_raw              str
text_raw                     str
rooms                    float64
area                     float64
number                   float64
price                    float64
dtype: object


## Count and remove missing values

In [30]:
# Count missing values
print(df.isna().sum())


# Drop rows with missing values (if any)
df_without_missing_values = df.dropna().copy()



web-scraper-order         0
web-scraper-start-url     0
rooms_area_price_raw      0
address_raw               0
price_raw                 0
description_raw           0
text_raw                  0
rooms                    45
area                     92
number                   15
price                    15
dtype: int64


## Count and remove duplicated values

In [33]:
# Count duplicated values


# Drop rows with duplicated values (if any)


# Dimension (rows, columns)


print("Anzahl Duplikate:", df.duplicated().sum())
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Anzahl Zeilen nach Entfernen der Duplikate:", len(df))

Anzahl Duplikate: 0
Anzahl Zeilen nach Entfernen der Duplikate: 1008


## Use string manipulation methods to create additional variables from the apartment descriptions.

### Change strings in 'description_raw' and 'text_raw' to uppercase 

In [11]:
# Change strings in 'description_raw' to uppercase


# Change strings in 'text_raw' to uppercase


### Calculate length of strings in 'description_raw' and 'text_raw'

In [12]:
# Length of the strings in 'description_raw'


# Length of the strings in 'text_raw'


# Histogram of the length of the strings in 'description_raw' and 'text_raw'


# Set the y-axis range for each subplot


# Display the plot


### Create binary (0/1) variable 'luxurious'

In [13]:
# Create a pattern which can be used to search the variable 'description_raw'


# Create new variable 'luxurious' as binary dummy (0/1) variable


# Count the number of luxurious apartments


# Show values


### Create variable 'price_per_m2'

In [14]:
# Create the new variable


# Show values


### Include current datetime as time stamp

In [15]:
# Get and format datetime


# Show values


## Discretization of numerical data

In [16]:
# Discretization of the variable 'price_per_m2'


# Show unique values


## One Hot Encoding

In [17]:
# One Hot Encoding of the variable 'price_per_m2_cat'


# Convert boolean values to integers (0 and 1)


# Show values of the new dummy variables


## Scaling

### Min-Max Scaling

#### By hand

In [18]:
# Min-Max scaling of the variable 'area' (by hand)


# Histogram of the variable 'area' and 'area_scaled_minmax'


# Set the y-axis range for each subplot and add labels


# Display the plot


#### scikit-learn version

In [19]:
# Min-Max scaling of the variable 'area' (scikit-learn version)


# Fit the scaler and transform the variable
# (note the double brackets: scikit-learn expects a 2D input)


# Check that the by-hand and scikit-learn versions give the same result


### Max-Absolute Scaling

#### By hand

In [20]:
# Max-Absolute scaling of the variable 'area' (by hand)


# Histogram of the variable 'area' and 'area_scaled_max_abs'


# Set the y-axis range for each subplot and add labels


# Display the plot


#### scikit-learn version

In [21]:
# Max-Absolute scaling of the variable 'area' (scikit-learn version)


# Fit the scaler and transform the variable
# (note the double brackets: scikit-learn expects a 2D input)


# Check that the by-hand and scikit-learn versions give the same result


### Robust Scaling

#### By hand

In [22]:
# Robust scaling of the variable 'area' (by hand)


# Histogram of the variable 'area' and 'area_scaled_robust'


# Set the y-axis range for each subplot and add labels


# Display the plot


#### scikit-learn version

In [23]:
# Robust scaling of the variable 'area' (scikit-learn version)


# Fit the scaler and transform the variable
# (note the double brackets: scikit-learn expects a 2D input)


# Check that the by-hand and scikit-learn versions give the same result


## Standardization

### Z-score Normalization

#### By hand

In [24]:
# Z-score Normalization of the variable 'area' (by hand)


# Histogram of the variable 'area' and 'area_scaled_z'


# Set the y-axis range for each subplot and add labels


# Display the plot


#### scikit-learn version

In [25]:
# Z-score Normalization of the variable 'area' (scikit-learn version)


# Fit the scaler and transform the variable
# (note the double brackets: scikit-learn expects a 2D input)


# Note: pandas .std() divides by n-1, StandardScaler divides by n,
# so the results differ slightly


# Check against the by-hand formula with the standard deviation
# divided by n (ddof=0)


## Transformation

### Log-Transformation

In [26]:
# Log-Transformation of the variable 'area'


# Histogram of the variable 'area' and 'area_log'


# Set the y-axis range for each subplot and add labels


# Display the plot


### SQRT-Transformation

In [27]:
# SQRT-Transformation of the variable 'area'


# Histogram of the variable 'area' and 'area_sqrt'


# Set the y-axis range for each subplot and add labels


# Display the plot


### Box-Cox Transformation

In [28]:
# Import the boxcox function from scipy


# Box-Cox Transformation of the variable 'area'


# Histogram of the variable 'area' and 'area_boxcox'


# Set the y-axis range for each subplot and add labels


# Display the plot


## Combining & organizing data

### Reading rental apartment data with geocoded addresses

In [29]:
# Meaning of variables
# lat: geographical latitude
# lon: geographical longitude
# bfs_number: official municipality id
# bfs_name: official municipality name

# Geocoded data (i.e. data with latitude and longitude)


# Show data


### Join geo-information to rental apartment data using .merge()

In [30]:
# Merge geo-information to the apartment data (the key is 'web-scraper-order')


# Show data


### Reading municipality-level data

In [31]:
# Meaning of variables:
# bfs_number: municipality id
# bfs_name: municipality name
# pop: number of residents
# pop_dens: population density per km2
# frg_pct: percentage foreigners
# emp: number of employees

# Read municipality-level data from the .xlsx file


# Show data


### Join municipality data to rental apartment data using .merge()

In [32]:
# Merge needs a key which must be identical in both data sets
# (here the key is 'bfs_number')


# Show data


## Sorting data

In [33]:
# Sorting data by 'price' and 'area' with highest price above (ascending=False)


# Show data


## Reshaping data

### Reshaping data using .stack() and .unstack()

In [34]:
# Create a subset of the data (first 5 rows)


# Reshape the data using .stack()


# Reshape the data back using .unstack()


### Reshaping data using .melt()

In [35]:
# Create a subset of the data (first 5 rows)


# Reshape the data using .melt()


### Pivoting data using .pivot_table()

In [36]:
# Create a pivot table with the mean values per number of rooms


# Format the values to 2 decimal places with trailing zeros


# Display the pivot table


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [37]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-11 14:32:47
Python Version: 3.13.5
-----------------------------------
